# 🎙️ WhisperX Audio Transcription — Multi-Speaker (Google Drive)

Notebook ini melakukan transkripsi audio/video secara otomatis dengan deteksi siapa yang berbicara (*speaker diarization*).

---

## ✅ Fitur
- Transkripsi berbasis **Whisper large-v3** (akurat untuk Bahasa Indonesia)
- Deteksi pembicara: `[Speaker 0]`, `[Speaker 1]`, dst.
- Membaca audio/video dari **Google Drive**
- Export ke **5 format**: `.txt`, `.md`, `.srt`, `.vtt`, `.json`

---

## 📋 Alur Kerja
| # | Cell | Keterangan |
|---|------|------------|
| 1 | Install Dependencies | Install ffmpeg, WhisperX, pyannote |
| 2 | Mount Google Drive | Hubungkan Drive ke Colab |
| 3 | Konfigurasi | Atur folder, model, dan parameter |
| 4 | HuggingFace Token | Login untuk akses model diarization |
| 5 | Load Models | Muat model ASR, alignment, diarization |
| 6 | Helper Functions | Fungsi bantu (tidak perlu diubah) |
| 7 | Cek File Audio | Lihat daftar file yang akan diproses |
| 8 | Transkripsi & Export | Proses utama |
| 9 | Ringkasan Output | Tampilkan lokasi semua file hasil |

---

## ⚠️ Prasyarat

### 1. Aktifkan GPU
```
Runtime → Change runtime type → T4 GPU  (atau L4 / A100)
```

### 2. HuggingFace Token
Diarization memerlukan akses ke model pyannote yang terkunci lisensi.

1. Buat token di [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Setujui syarat penggunaan model berikut:
   - [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1)
   - [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1)
   - [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0)
3. Simpan token di **Colab Secrets** (ikon 🔑 di sidebar) dengan nama `HF_TOKEN`


In [ ]:
# ─── CELL 1: Install Dependencies ───────────────────────────────────────────
# Jalankan sekali. Setelah selesai, RESTART SESSION lalu lanjut ke Cell 2.
# Urutan instalasi ini penting untuk menghindari konflik versi di Colab.

!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Hapus versi lama yang mungkin konflik
!pip -q uninstall -y whisperx pyannote.audio pyannote.core pyannote.metrics huggingface_hub pandas numpy

# Install versi yang kompatibel
# NumPy 1.26.4 = versi 1.x terakhir yang stabil, menghindari error '_center'
!pip -q install "numpy==1.26.4" "pandas==2.2.2" "huggingface-hub>=0.34.0,<1.0"
!pip -q install git+https://github.com/m-bain/whisperx.git

# Paksa versi NumPy & Pandas agar tidak di-override oleh sub-dependency
!pip -q install --upgrade --force-reinstall --no-deps "numpy==1.26.4" "pandas==2.2.2"

import numpy, pandas, huggingface_hub
print("✅ Instalasi selesai.")
print(f"   NumPy   : {numpy.__version__}")
print(f"   Pandas  : {pandas.__version__}")
print()
print("⚠️  PENTING: Sekarang klik Runtime → Restart session, lalu jalankan mulai Cell 2.")


In [ ]:
# ─── CELL 2: Mount Google Drive ─────────────────────────────────────────────

from google.colab import drive

drive.mount("/content/drive")
print("✅ Google Drive terhubung.")


In [ ]:
# ─── CELL 3: Konfigurasi ────────────────────────────────────────────────────
# Sesuaikan bagian ini sebelum menjalankan proses transkripsi.

import os
import re
import json
from pathlib import Path
from collections import Counter

import torch
from huggingface_hub import login

# Bersihkan VRAM dari sesi sebelumnya yang mungkin gagal
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =============================================================================
# 📁 FOLDER
# =============================================================================
AUDIO_FOLDER  = "/content/drive/MyDrive/Whisper/AudioFiles"            # Folder file audio/video input
OUTPUT_FOLDER = "/content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker"  # Folder hasil transkripsi

# =============================================================================
# 🤖 MODEL & BAHASA
# =============================================================================
MODEL_NAME = "large-v3"  # Opsi: tiny, base, small, medium, large-v2, large-v3
LANGUAGE   = "id"        # Kode bahasa ISO 639-1. Contoh: "id" (Indonesia), "en" (Inggris)

# =============================================================================
# ⚙️ PERFORMA
# =============================================================================
# Turunkan BATCH_SIZE jika muncul error "Out of Memory" pada GPU T4
BATCH_SIZE = 4

# =============================================================================
# 👥 SPEAKER DIARIZATION
# =============================================================================
# Isi angka jika kamu tahu persis jumlah pembicara. Biarkan None untuk deteksi otomatis.
MIN_SPEAKERS = None
MAX_SPEAKERS = None

# Gabungkan kalimat berurutan dari speaker yang sama agar lebih mudah dibaca
MERGE_SAME_SPEAKER    = True
MERGE_MAX_GAP_SECONDS = 1.25   # Jeda maksimal (detik) antar segment yang masih digabung
MERGE_MAX_CHARS       = 1800   # Batas panjang karakter per segment hasil gabungan

# =============================================================================
# 🔁 LAIN-LAIN
# =============================================================================
# Lewati file yang sudah ada hasil transkripsinya (berguna jika ingin lanjut dari tengah)
SKIP_EXISTING = False

SUPPORTED_EXTENSIONS = [
    ".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".wma",
    ".mp4", ".mov", ".mkv", ".webm"
]

# Prompt awal untuk membantu Whisper memahami konteks audio
INITIAL_PROMPT = """
Transkripsi Bahasa Indonesia yang rapi dan akurat.
Pertahankan nama orang, tempat, lembaga, dan istilah penting dengan ejaan yang benar.
"""

# =============================================================================
# Deteksi device & buat folder jika belum ada
# =============================================================================
os.makedirs(AUDIO_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print("✅ Konfigurasi siap.")
print(f"   Device      : {DEVICE}")
print(f"   Model       : {MODEL_NAME}")
print(f"   Bahasa      : {LANGUAGE}")
print(f"   Batch Size  : {BATCH_SIZE}")
print(f"   Input       : {AUDIO_FOLDER}")
print(f"   Output      : {OUTPUT_FOLDER}")


In [ ]:
# ─── CELL 4: HuggingFace Token ──────────────────────────────────────────────
# Model diarization pyannote memerlukan autentikasi HuggingFace.
#
# Cara menyimpan token dengan aman:
#   1. Klik ikon 🔑 (Secrets) di sidebar kiri Colab
#   2. Klik "Add new secret"
#   3. Name: HF_TOKEN  |  Value: <token kamu>
#   4. Aktifkan toggle "Notebook access"

HF_TOKEN = None

# Coba ambil dari Colab Secrets terlebih dahulu
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

# Fallback: minta input manual
if not HF_TOKEN:
    HF_TOKEN = input("Token tidak ditemukan di Secrets. Paste HuggingFace token kamu: ").strip()

if not HF_TOKEN:
    raise ValueError("HF_TOKEN kosong. Diarization tidak bisa berjalan tanpa token HuggingFace.")

login(token=HF_TOKEN)
print("✅ HuggingFace login berhasil.")


In [ ]:
# ─── CELL 5: Load Models ────────────────────────────────────────────────────
# Memuat 3 model sekaligus: ASR (transkripsi), alignment (sinkronisasi kata),
# dan diarization (deteksi pembicara).
# Proses ini butuh beberapa menit pada pertama kali (download model).

import whisperx

# Pastikan token tersedia untuk library yang membutuhkannya via env variable
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

# --- Model 1: ASR (Speech-to-Text) ---
print(f"⏳ Memuat model ASR: {MODEL_NAME} ...")
try:
    asr_model = whisperx.load_model(
        MODEL_NAME,
        DEVICE,
        compute_type=COMPUTE_TYPE,
        language=LANGUAGE,
        asr_options={"initial_prompt": INITIAL_PROMPT, "condition_on_previous_text": True}
    )
except TypeError:
    # Fallback untuk versi WhisperX lama yang belum mendukung asr_options
    asr_model = whisperx.load_model(MODEL_NAME, DEVICE, compute_type=COMPUTE_TYPE, language=LANGUAGE)
print("✅ Model ASR siap.")

# --- Model 2: Word Alignment ---
print("⏳ Memuat model alignment ...")
align_model, align_metadata = whisperx.load_align_model(language_code=LANGUAGE, device=DEVICE)
print("✅ Model alignment siap.")

# --- Model 3: Speaker Diarization ---
print("⏳ Memuat model diarization ...")
try:
    diarize_model = whisperx.diarize.DiarizationPipeline(
        model_name="pyannote/speaker-diarization-3.1",
        device=DEVICE
    )
    print("✅ Model diarization siap.")
except Exception as e:
    print(f"❌ Gagal memuat model diarization: {e}")
    print("   Pastikan kamu sudah menyetujui syarat model di:")
    print("   https://huggingface.co/pyannote/speaker-diarization-3.1")

print()
print("✅ Semua model siap digunakan.")


In [ ]:
# ─── CELL 6: Helper Functions ───────────────────────────────────────────────
# Fungsi-fungsi bantu untuk formatting, normalisasi, dan export.
# Tidak perlu diubah.

def format_timestamp_comma(seconds: float) -> str:
    """Mengubah detik menjadi format SRT: 00:00:00,100"""
    if seconds is None:
        seconds = 0.0
    total_ms = int(round(float(seconds) * 1000))
    h  = total_ms // 3_600_000
    m  = (total_ms % 3_600_000) // 60_000
    s  = (total_ms % 60_000) // 1000
    ms = total_ms % 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def format_timestamp_dot(seconds: float) -> str:
    """Mengubah detik menjadi format VTT: 00:00:00.100"""
    return format_timestamp_comma(seconds).replace(",", ".")


def clean_text(text: str) -> str:
    """Hapus whitespace berlebih."""
    if not text:
        return ""
    return " ".join(text.strip().split())


def normalize_speaker_label(raw_speaker) -> str:
    """Ubah label pyannote (SPEAKER_00) menjadi format yang lebih bersih (Speaker 0)."""
    if raw_speaker is None:
        return "Speaker ?"
    match = re.search(r"(\d+)$", str(raw_speaker))
    if match:
        return f"Speaker {int(match.group(1))}"
    return str(raw_speaker).replace("_", " ").title()


def speaker_from_words(segment) -> str:
    """Ambil speaker paling dominan dari label per-kata (fallback jika label segment kosong)."""
    words = segment.get("words", []) or []
    speakers = [w.get("speaker") for w in words if w.get("speaker")]
    if speakers:
        return Counter(speakers).most_common(1)[0][0]
    return segment.get("speaker")


def normalize_segments(result_segments: list) -> list:
    """Ubah output WhisperX menjadi list segment standar: {id, start, end, speaker, text}."""
    normalized = []
    for i, seg in enumerate(result_segments):
        text = clean_text(seg.get("text", ""))
        if not text:
            continue
        raw_speaker = seg.get("speaker") or speaker_from_words(seg)
        normalized.append({
            "id":      i,
            "start":   float(seg.get("start", 0.0)),
            "end":     float(seg.get("end",   0.0)),
            "speaker": normalize_speaker_label(raw_speaker),
            "text":    text
        })
    return normalized


def merge_same_speaker_segments(segments: list, max_gap: float = 1.25, max_chars: int = 1800) -> list:
    """Gabungkan segment berurutan dari speaker yang sama agar lebih mudah dibaca."""
    if not segments:
        return []
    merged  = []
    current = dict(segments[0])
    for seg in segments[1:]:
        same_speaker  = seg["speaker"] == current["speaker"]
        gap           = seg["start"] - current["end"]
        combined_len  = len(current["text"]) + 1 + len(seg["text"])
        if same_speaker and gap <= max_gap and combined_len <= max_chars:
            current["end"]  = seg["end"]
            current["text"] = clean_text(current["text"] + " " + seg["text"])
        else:
            merged.append(current)
            current = dict(seg)
    merged.append(current)
    for idx, seg in enumerate(merged):
        seg["id"] = idx
    return merged


def find_audio_files(folder_path: str) -> list:
    """Cari semua file audio/video yang didukung di dalam folder (termasuk subfolder)."""
    folder = Path(folder_path)
    files  = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS]
    return sorted(files)


def save_txt(segments: list, output_path: str):
    """Simpan transkrip format TXT dengan timestamp dan label speaker."""
    with open(output_path, "w", encoding="utf-8") as f:
        for seg in segments:
            text = clean_text(seg["text"])
            if not text:
                continue
            f.write(f"{format_timestamp_comma(seg['start'])} --> {format_timestamp_comma(seg['end'])} [{seg['speaker']}]\n")
            f.write(f"{text}\n\n")


def save_markdown(segments: list, output_path: str, title: str = "Transkrip Audio"):
    """Simpan transkrip format Markdown."""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")
        for seg in segments:
            text = clean_text(seg["text"])
            if not text:
                continue
            f.write(f"### {format_timestamp_comma(seg['start'])} → {format_timestamp_comma(seg['end'])} [{seg['speaker']}]\n\n")
            f.write(f"{text}\n\n")


def save_srt(segments: list, output_path: str, include_speaker: bool = True):
    """Simpan transkrip format SRT (subtitle)."""
    with open(output_path, "w", encoding="utf-8") as f:
        index = 1
        for seg in segments:
            text = clean_text(seg["text"])
            if not text:
                continue
            if include_speaker:
                text = f"[{seg['speaker']}] {text}"
            f.write(f"{index}\n")
            f.write(f"{format_timestamp_comma(seg['start'])} --> {format_timestamp_comma(seg['end'])}\n")
            f.write(f"{text}\n\n")
            index += 1


def save_vtt(segments: list, output_path: str, include_speaker: bool = True):
    """Simpan transkrip format WebVTT (subtitle untuk web)."""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")
        for seg in segments:
            text = clean_text(seg["text"])
            if not text:
                continue
            if include_speaker:
                text = f"[{seg['speaker']}] {text}"
            f.write(f"{format_timestamp_dot(seg['start'])} --> {format_timestamp_dot(seg['end'])}\n")
            f.write(f"{text}\n\n")


def save_json(segments: list, raw_result: dict, output_path: str):
    """Simpan transkrip + metadata ke format JSON."""
    data = {
        "model":        MODEL_NAME,
        "language":     LANGUAGE,
        "min_speakers": MIN_SPEAKERS,
        "max_speakers": MAX_SPEAKERS,
        "segments":     segments,
        "raw_segments": raw_result.get("segments", [])
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


print("✅ Helper functions siap.")


In [ ]:
# ─── CELL 7: Cek File Audio ─────────────────────────────────────────────────
# Tampilkan semua file audio/video yang ditemukan di AUDIO_FOLDER.
# Pastikan semua file yang ingin diproses sudah ada sebelum lanjut.

audio_files = find_audio_files(AUDIO_FOLDER)

print(f"📂 Folder : {AUDIO_FOLDER}")
print(f"📄 Total  : {len(audio_files)} file ditemukan")
print()

if not audio_files:
    print("⚠️  Tidak ada file audio/video yang ditemukan.")
    print("   Upload file ke folder tersebut lalu jalankan cell ini lagi.")
else:
    for i, file in enumerate(audio_files, start=1):
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  {i:2}. {file.name:<50} ({size_mb:.1f} MB)")


In [ ]:
# ─── CELL 8: Transkripsi, Diarization & Export ──────────────────────────────
# Proses utama. Setiap file akan:
#   1. Ditranskripsi oleh Whisper
#   2. Diselaraskan kata per kata
#   3. Dideteksi speaker-nya
#   4. Diekspor ke TXT, MD, SRT, VTT, JSON

def transcribe_file(audio_path: Path) -> dict:
    print("=" * 70)
    print(f"🎧 File    : {audio_path.name}")
    print(f"   Ukuran  : {audio_path.stat().st_size / (1024*1024):.1f} MB")

    base_name = audio_path.stem
    out = {
        "audio": str(audio_path),
        "txt":   os.path.join(OUTPUT_FOLDER, f"{base_name}.txt"),
        "md":    os.path.join(OUTPUT_FOLDER, f"{base_name}.md"),
        "srt":   os.path.join(OUTPUT_FOLDER, f"{base_name}.srt"),
        "vtt":   os.path.join(OUTPUT_FOLDER, f"{base_name}.vtt"),
        "json":  os.path.join(OUTPUT_FOLDER, f"{base_name}.json"),
    }

    if SKIP_EXISTING and os.path.exists(out["txt"]):
        print("⏭️  Dilewati (output sudah ada).")
        return {**out, "skipped": True}

    # Step 1: Transkripsi
    print("   [1/4] Memuat audio...")
    audio = whisperx.load_audio(str(audio_path))

    print("   [2/4] Transkripsi...")
    result = asr_model.transcribe(audio, batch_size=BATCH_SIZE, language=LANGUAGE)

    # Step 2: Word Alignment
    print("   [3/4] Alignment kata-timestamp...")
    result = whisperx.align(
        result["segments"], align_model, align_metadata,
        audio, DEVICE, return_char_alignments=False
    )

    # Step 3: Speaker Diarization
    print("   [4/4] Deteksi pembicara (diarization)...")
    diarize_kwargs = {}
    if MIN_SPEAKERS is not None:
        diarize_kwargs["min_speakers"] = MIN_SPEAKERS
    if MAX_SPEAKERS is not None:
        diarize_kwargs["max_speakers"] = MAX_SPEAKERS

    diarize_segments = diarize_model(str(audio_path), **diarize_kwargs)
    result = whisperx.assign_word_speakers(diarize_segments, result)

    # Step 4: Normalisasi & Merge
    segments = normalize_segments(result["segments"])
    if MERGE_SAME_SPEAKER:
        segments = merge_same_speaker_segments(
            segments, max_gap=MERGE_MAX_GAP_SECONDS, max_chars=MERGE_MAX_CHARS
        )

    # Preview 20 segment pertama
    print()
    print(f"   Preview ({min(20, len(segments))} dari {len(segments)} segment):")
    for seg in segments[:20]:
        ts = f"{format_timestamp_comma(seg['start'])} → {format_timestamp_comma(seg['end'])}"
        print(f"   {ts} [{seg['speaker']}] {seg['text'][:80]}")
    if len(segments) > 20:
        print(f"   ... dan {len(segments) - 20} segment lainnya")

    # Export semua format
    save_txt(segments, out["txt"])
    save_markdown(segments, out["md"], title=base_name)
    save_srt(segments, out["srt"])
    save_vtt(segments, out["vtt"])
    save_json(segments, result, out["json"])

    print()
    print("   ✅ Output tersimpan:")
    for fmt, path in [("TXT", out["txt"]), ("MD", out["md"]), ("SRT", out["srt"]),
                      ("VTT", out["vtt"]), ("JSON", out["json"])]:
        print(f"      {fmt:<5}: {path}")

    return {**out, "skipped": False}


# ─── Jalankan untuk semua file ───────────────────────────────────────────────
results = []

if not audio_files:
    print("⚠️  Tidak ada file audio/video. Jalankan Cell 7 terlebih dahulu.")
else:
    for audio_file in audio_files:
        try:
            result = transcribe_file(audio_file)
            results.append(result)
        except Exception as e:
            print(f"\n❌ Error pada file: {audio_file.name}")
            print(f"   {type(e).__name__}: {e}")
            print()
            print("   💡 Tips:")
            print("   - VRAM habis? Turunkan BATCH_SIZE di Cell 3 (coba nilai 2 atau 1).")
            print("   - Token salah? Ulangi Cell 4.")
            print("   - Belum accept terms pyannote? Buka link di Cell 4.")
            print("   - Speaker terlalu banyak? Set MAX_SPEAKERS di Cell 3.")

print()
print("=" * 70)
processed = sum(1 for r in results if not r.get("skipped"))
skipped   = sum(1 for r in results if r.get("skipped"))
print(f"✅ Selesai! Diproses: {processed} file  |  Dilewati: {skipped} file")


In [ ]:
# ─── CELL 9: Ringkasan Output ───────────────────────────────────────────────
# Tampilkan lokasi semua file hasil transkripsi.

if not results:
    print("Belum ada hasil. Jalankan Cell 8 terlebih dahulu.")
else:
    print(f"📁 Output tersimpan di: {OUTPUT_FOLDER}")
    print()
    for item in results:
        status = "⏭️ SKIP" if item.get("skipped") else "✅ DONE"
        print(f"{status}  {Path(item['audio']).name}")
        if not item.get("skipped"):
            for fmt, key in [("TXT", "txt"), ("MD", "md"), ("SRT", "srt"), ("VTT", "vtt"), ("JSON", "json")]:
                print(f"        {fmt:<5}: {item[key]}")
        print()


---

## 📄 Contoh Output `.txt`

```
00:00:00,100 --> 00:00:33,940 [Speaker 0]
Selamat pagi dan selamat datang di acara hari ini.

00:00:33,940 --> 00:03:27,580 [Speaker 1]
Terima kasih. Saya senang bisa hadir di sini untuk membahas topik yang sangat penting ini.

00:03:27,800 --> 00:03:34,500 [Speaker 2]
Pertanyaan pertama dari saya adalah...
```

---

## 🔧 Troubleshooting

| Masalah | Solusi |
|---------|--------|
| **CUDA out of memory** | Turunkan `BATCH_SIZE` di Cell 3 (coba `2` atau `1`) |
| **Speaker terlalu banyak/sedikit** | Set `MIN_SPEAKERS` dan `MAX_SPEAKERS` di Cell 3 |
| **Diarization gagal load** | Pastikan sudah accept terms di HuggingFace (link di Cell 4) |
| **Nama speaker bukan nama asli** | Normal — diarization mengenali karakter suara, bukan nama orang |
| **Akurasi rendah** | Pastikan `LANGUAGE` sesuai, atau coba `BATCH_SIZE` lebih kecil |
